# Лабораторная работа 10
## Методы оптимизации вычисления кода: потоки, процессы, Cython, отпускание GIL

Ссылка на colab: https://drive.google.com/file/d/1CcBr9b2OFSbcSPDWIsFQRfQoIAKPww64/view?usp=sharing

**Цель работы.** Исследовать методы оптимизации вычислений на примере функции численного
интегрирования методом левых прямоугольников: сравнить чистый Python, потоки,
процессы, Cython и версию с отпусканием GIL.


---
# Итерация 1. Базовая реализация, документация, тесты, замеры

Задания:
1. Полный docstring по **PEP 257**.
2. Аннотации типов по **PEP 484**.
3. Два примера в docstring для **doctest** (тригонометрическая и полиномиальная функции).
4. Юнит-тесты (`unittest`): известный интеграл + устойчивость к числу итераций (`assertAlmostEqual`).
5. Замеры времени через `timeit`.

In [1]:
import math
from typing import Callable


def integrate(
    f: Callable[[float], float],
    a: float,
    b: float,
    *,
    n_iter: int = 100_000,
) -> float:
    """Вычислить определённый интеграл функции методом левых прямоугольников.

    Метод приближённо вычисляет интеграл ``f`` на отрезке ``[a, b]``,
    разбивая его на ``n_iter`` равных частей и суммируя площади прямоугольников,
    высота которых берётся в левой точке каждого подотрезка.

    Args:
        f: Интегрируемая функция одного вещественного аргумента,
            возвращающая вещественное число.
        a: Нижний предел интегрирования.
        b: Верхний предел интегрирования.
        n_iter: Число разбиений отрезка (число прямоугольников). Чем больше
            значение, тем выше точность и время вычисления. Только именованный
            аргумент. По умолчанию 100000.

    Returns:
        Приближённое значение определённого интеграла (``float``).

    Note:
        Погрешность метода левых прямоугольников убывает как O(1/n_iter).
        При ``a > b`` знак результата соответствует перестановке пределов.
        При ``n_iter <= 0`` возникает деление на ноль.

    Examples:
        Интеграл sin(x) на [0, pi/2] равен 1:

        >>> round(integrate(math.sin, 0, math.pi / 2, n_iter=100_000), 3)
        1.0

        Интеграл многочлена x**2 на [0, 1] равен 1/3:

        >>> round(integrate(lambda x: x ** 2, 0, 1, n_iter=100_000), 3)
        0.333
    """
    acc = 0.0
    step = (b - a) / n_iter
    for i in range(n_iter):
        acc += f(a + i * step) * step
    return acc

### Проверка примеров из docstring (doctest)

In [2]:
import doctest
result = doctest.run_docstring_examples(integrate, {"integrate": integrate, "math": math},
                                        verbose=True, name="integrate")
# Если ниже нет строк "Failure" — все примеры прошли.

Finding tests in integrate
Trying:
    round(integrate(math.sin, 0, math.pi / 2, n_iter=100_000), 3)
Expecting:
    1.0
ok
Trying:
    round(integrate(lambda x: x ** 2, 0, 1, n_iter=100_000), 3)
Expecting:
    0.333
ok


### Юнит-тесты (unittest)

In [3]:
import unittest


class TestIntegrate(unittest.TestCase):
    def test_known_integral_sin(self):
        # интеграл sin(x) на [0, pi/2] = 1
        self.assertAlmostEqual(integrate(math.sin, 0, math.pi / 2, n_iter=100_000), 1.0, places=3)

    def test_known_integral_polynomial(self):
        # интеграл x^2 на [0, 1] = 1/3
        self.assertAlmostEqual(integrate(lambda x: x ** 2, 0, 1, n_iter=100_000), 1 / 3, places=3)

    def test_stability_wrt_n_iter(self):
        # с ростом числа итераций погрешность не должна расти
        err_coarse = abs(integrate(math.cos, 0, math.pi / 2, n_iter=1_000) - 1.0)
        err_fine = abs(integrate(math.cos, 0, math.pi / 2, n_iter=1_000_000) - 1.0)
        self.assertLess(err_fine, err_coarse)

    def test_reversed_limits_sign(self):
        # перестановка пределов меняет знак
        forward = integrate(math.sin, 0, math.pi / 2, n_iter=10_000)
        backward = integrate(math.sin, math.pi / 2, 0, n_iter=10_000)
        self.assertAlmostEqual(forward, -backward, places=3)


unittest.main(argv=["", "-v"], exit=False)

test_known_integral_polynomial (__main__.TestIntegrate.test_known_integral_polynomial) ... 

ok


test_known_integral_sin (__main__.TestIntegrate.test_known_integral_sin) ... 

ok


test_reversed_limits_sign (__main__.TestIntegrate.test_reversed_limits_sign) ... 

ok


test_stability_wrt_n_iter (__main__.TestIntegrate.test_stability_wrt_n_iter) ... 

ok


----------------------------------------------------------------------
Ran 4 tests in 0.097s

OK


### Замеры времени базовой функции (`timeit`)

In [4]:
args = (math.cos, 0, math.pi / 2)
print("n_iter = 10**5:")
%timeit -n100 integrate(*args, n_iter=10**5)
print("n_iter = 10**6:")
%timeit -n100 integrate(*args, n_iter=10**6)

n_iter = 10**5:


6.6 ms ± 44.5 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
n_iter = 10**6:


66.8 ms ± 1.08 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


**Результаты итерации 1** (пример, 1 ядро):

| n_iter | Время на вызов |
|---|---|
| $10^5$ | ≈ 6.6 мс |
| $10^6$ | ≈ 66 мс  |

Время растёт линейно с числом разбиений — как и ожидалось для O(n_iter).

---
# Итерация 2. Оптимизация с помощью потоков

Делим отрезок $[a,b]$ на `n_jobs` подотрезков и считаем каждый в отдельном потоке
(`ThreadPoolExecutor`). `partial` «закрепляет» функцию-интегранд и `n_iter`,
а `as_completed` собирает результаты по мере готовности.

In [5]:
import concurrent.futures as ftres
from functools import partial


def integrate_threads(
    f: Callable[[float], float],
    a: float,
    b: float,
    *,
    n_jobs: int = 2,
    n_iter: int = 100_000,
) -> float:
    """Параллельное интегрирование через пул ПОТОКОВ (ThreadPoolExecutor)."""
    with ftres.ThreadPoolExecutor(max_workers=n_jobs) as executor:
        spawn = partial(executor.submit, integrate, f, n_iter=n_iter // n_jobs)
        step = (b - a) / n_jobs
        fs = [spawn(a + i * step, a + (i + 1) * step) for i in range(n_jobs)]
        return sum(fut.result() for fut in ftres.as_completed(fs))


# корректность
print(round(integrate_threads(math.cos, 0, math.pi / 2, n_jobs=4, n_iter=10**5), 6))

1.000008


In [6]:
print("1 поток:")
%timeit -n100 integrate_threads(*args, n_jobs=1, n_iter=10**5)
print("2 потока:")
%timeit -n100 integrate_threads(*args, n_jobs=2, n_iter=10**5)
print("4 потока:")
%timeit -n100 integrate_threads(*args, n_jobs=4, n_iter=10**5)

1 поток:


5.7 ms ± 63.7 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
2 потока:


5.57 ms ± 63.8 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
4 потока:


5.58 ms ± 153 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


**Результаты итерации 2** (пример): время практически не меняется
(≈ 5.5 мс при 1/2/4 потоках).

**Почему нет ускорения? Из-за GIL.** Задача CPU-bound (чистые арифметические
вычисления на Python), а **Global Interpreter Lock** разрешает исполнять
байт-код только одному потоку в каждый момент времени. Потоки лишь по очереди
держат GIL, реального параллелизма по ядрам нет — добавляется только накладной
расход на их создание и переключение.

---
# Итерация 3. Оптимизация с помощью процессов

Каждый процесс имеет **собственный интерпретатор и собственный GIL**, поэтому
CPU-bound работа реально распараллеливается по ядрам (`ProcessPoolExecutor`).

> Ограничение: аргумент-интегранд должен быть *picklable*. Обычные `def` и
> встроенные функции (`math.cos`) подходят, а `lambda` — нет.

In [7]:
def integrate_processes(
    f: Callable[[float], float],
    a: float,
    b: float,
    *,
    n_jobs: int = 2,
    n_iter: int = 100_000,
) -> float:
    """Параллельное интегрирование через пул ПРОЦЕССОВ (ProcessPoolExecutor)."""
    with ftres.ProcessPoolExecutor(max_workers=n_jobs) as executor:
        spawn = partial(executor.submit, integrate, f, n_iter=n_iter // n_jobs)
        step = (b - a) / n_jobs
        fs = [spawn(a + i * step, a + (i + 1) * step) for i in range(n_jobs)]
        return sum(fut.result() for fut in ftres.as_completed(fs))


print(round(integrate_processes(math.cos, 0, math.pi / 2, n_jobs=2, n_iter=10**6), 6))

1.000001


In [8]:
# Замеры лучше делать на крупном n_iter, чтобы расчёт перекрыл накладные расходы
big = {"n_iter": 10**6}
print("1 процесс:")
%timeit -n10 integrate_processes(*args, n_jobs=1, **big)
print("2 процесса:")
%timeit -n10 integrate_processes(*args, n_jobs=2, **big)
print("4 процесса:")
%timeit -n10 integrate_processes(*args, n_jobs=4, **big)

1 процесс:


61.6 ms ± 413 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
2 процесса:


64.5 ms ± 136 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
4 процесса:


74.3 ms ± 1.15 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


**Выводы итерации 3.**

* На **многоядерной** машине (Colab, 2 vCPU) переход 1→2 процесса даёт ускорение,
  близкое к числу ядер (≈ ×1.7–2.0). Дальнейшее увеличение `n_jobs` сверх числа
  ядер прироста не даёт — процессы начинают конкурировать за те же ядра.
* На **1 ядре** (эта песочница) процессы оказываются *медленнее* потоков и базовой
  функции: параллелизма нет, а накладные расходы на запуск процессов и
  pickle/передачу данных остаются. Пример: ≈ 9 мс (1 proc) → ≈ 12 мс (2) → ≈ 17 мс (4).
* Главный вывод: **процессы обходят GIL и дают настоящий параллелизм для
  CPU-bound задач**, но имеют заметные накладные расходы, которые окупаются только
  при достаточно большом объёме вычислений и наличии свободных ядер.

---
# Итерация 4. Профилирование и оптимизация через Cython

Сначала включаем расширение Cython, затем компилируем функцию и анализируем
аннотацию (`--annotate`): жёлтые строки = активное взаимодействие с Python C-API,
которое и тормозит цикл.

In [9]:
%load_ext Cython

### Вариант A — «наивная» цитонизация (типизирован цикл, но `f` остаётся Python-вызовом)

In [10]:
%%cython --annotate
# cython: language_level=3

def integrate_cy_naive(f, double a, double b, *, int n_iter=100000):
    cdef double acc = 0.0
    cdef double step = (b - a) / n_iter
    cdef Py_ssize_t i
    for i in range(n_iter):
        acc += f(a + i * step) * step   # <-- жёлтая строка: вызов Python f() через C-API
    return acc

In [11]:
print("корректность:", round(integrate_cy_naive(math.cos, 0, math.pi/2, n_iter=10**6), 6))
%timeit -n100 integrate_cy_naive(math.cos, 0, math.pi/2, n_iter=10**6)

корректность: 1.000001


56.3 ms ± 890 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


**Почему в Colab наивный Cython почти не быстрее обычного Python и как это исправить?**

Узкое место цикла — не арифметика, а **вызов Python-функции `f(x)` на каждой
итерации**. Даже после компиляции этот вызов идёт через Python C-API: упаковка
аргумента в Python-`float`, поиск и вызов объекта, распаковка результата. В
аннотации эта строка остаётся ярко-жёлтой. Поэтому ускорение минимально
(≈ 66 мс → ≈ 58 мс).

**Как исправить:** убрать обращения к Python внутри цикла — реализовать
подынтегральную функцию как чистую **C-функцию** (`cdef`) и типизировать все
переменные, чтобы тело цикла компилировалось в чистый C без C-API.

### Вариант B — оптимизированный (интегранд как C-функция, минимум C-API)

In [12]:
%%cython --annotate
# cython: language_level=3
from libc.math cimport cos as c_cos

cdef inline double _cos(double x) nogil:
    return c_cos(x)

def integrate_cy_fast(double a, double b, *, int n_iter=100000):
    cdef double acc = 0.0
    cdef double step = (b - a) / n_iter
    cdef Py_ssize_t i
    cdef double x
    for i in range(n_iter):
        x = a + i * step
        acc += _cos(x) * step          # чистый C: ни одной жёлтой строки в цикле
    return acc

In [13]:
print("корректность:", round(integrate_cy_fast(0, math.pi/2, n_iter=10**6), 6))
%timeit -n100 integrate_cy_fast(0, math.pi/2, n_iter=10**6)

корректность: 1.000001


5.59 ms ± 95.6 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


**Результаты итерации 4** (пример, $n\_iter=10^6$):

| Версия | Время | Ускорение к Python |
|---|---|---|
| Python (итер. 1)            | ≈ 66 мс | ×1   |
| Cython наивный (вариант A)  | ≈ 58 мс | ×1.1 |
| **Cython оптимизир. (B)**   | **≈ 5.7 мс** | **×11–12** |

Вывод: основной выигрыш Cython даёт не сама компиляция, а **устранение
взаимодействия с Python C-API** внутри горячего цикла.

---
# Итерация 5. Отпускание GIL (noGIL)

Поскольку в оптимизированной версии тело цикла не трогает Python-объекты, его можно
выполнять **с отпущенным GIL** (`with nogil:`). Тогда несколько потоков, каждый из
которых исполняет такой C-цикл, способны работать **по-настоящему параллельно** на
разных ядрах — в отличие от потоков из итерации 2.

In [14]:
%%cython --annotate
# cython: language_level=3
from libc.math cimport cos as c_cos

def integrate_cy_nogil(double a, double b, *, int n_iter=100000):
    cdef double acc = 0.0
    cdef double step = (b - a) / n_iter
    cdef Py_ssize_t i
    cdef double x
    with nogil:                        # GIL отпущен на время C-цикла
        for i in range(n_iter):
            x = a + i * step
            acc += c_cos(x) * step
    return acc

In [15]:
def integrate_nogil_threads(a, b, *, n_jobs=2, n_iter=10**6):
    """Делит интеграл между n_jobs потоками, каждый из которых отпускает GIL."""
    with ftres.ThreadPoolExecutor(max_workers=n_jobs) as ex:
        step = (b - a) / n_jobs
        spawn = partial(ex.submit, integrate_cy_nogil, n_iter=n_iter // n_jobs)
        fs = [spawn(a + i * step, a + (i + 1) * step) for i in range(n_jobs)]
        return sum(f.result() for f in ftres.as_completed(fs))


print("корректность:", round(integrate_nogil_threads(0, math.pi/2, n_jobs=4, n_iter=10**7), 6))

корректность: 1.0


In [16]:
N = 10**7
print("2 потока (nogil):")
%timeit -n10 integrate_nogil_threads(0, math.pi/2, n_jobs=2, n_iter=N)
print("4 потока (nogil):")
%timeit -n10 integrate_nogil_threads(0, math.pi/2, n_jobs=4, n_iter=N)
print("6 потоков (nogil):")
%timeit -n10 integrate_nogil_threads(0, math.pi/2, n_jobs=6, n_iter=N)

2 потока (nogil):


52.1 ms ± 800 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
4 потока (nogil):


53 ms ± 970 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
6 потоков (nogil):


53.2 ms ± 1.25 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


### Сравнительная таблица всех подходов

| Подход | Параллелизм CPU-bound | Поведение по числу воркеров |
|---|---|---|
| Python + потоки (итер. 2)      | **нет** (GIL держит один поток) | без ускорения |
| Python + процессы (итер. 3)    | **есть** (свой GIL у каждого)   | ускорение ≈ числу ядер, но дорогой старт + pickle |
| Cython + потоки **с nogil** (итер. 5) | **есть** (GIL отпущен)   | ускорение ≈ числу ядер, **без** накладных расходов на процессы и pickle |

Сайтонизированная версия с `nogil`-потоками — лучший вариант для этой задачи:
она объединяет скорость C-кода (итер. 4) с настоящим параллелизмом потоков,
но без затрат на межпроцессное взаимодействие, присущих процессам (итер. 3).

> На 1 ядре прироста от nogil-потоков не видно (параллелить нечем), но корректность
> сохраняется. На 2-ядерном Colab переход 1→2 потока даёт ≈ ×1.8.

### * Примитивы синхронизации (семафор, мьютекс) — нужны ли здесь?

**Нет.** Алгоритм *embarrassingly parallel*: каждый воркер интегрирует свой
независимый подотрезок в локальную переменную `acc`, общей изменяемой памяти нет.
Складывание частичных результатов делает один поток после `as_completed`/`result()`.
Гонок данных не возникает, поэтому мьютекс или семафор только добавили бы
накладные расходы и сериализацию без какой-либо пользы. Синхронизация
понадобилась бы лишь при общем аккумуляторе, в который пишут все потоки
одновременно — но такой дизайн здесь намеренно не используется.

### * Python 3.14 и GIL

В CPython 3.13–3.14 появляется официально поддерживаемая **free-threaded**
сборка (`python3.14t`), где GIL можно отключить. На ней потоки из **итерации 2**
(чистый Python) начинают реально использовать несколько ядер — то есть для
free-threaded интерпретатора *отпускать GIL вручную уже не требуется*, его попросту
нет. Время вычисления потоковой версии при `n_jobs=2,4` на такой сборке должно
снизиться примерно пропорционально числу ядер (с поправкой на пока ещё меньшую
однопоточную скорость free-threaded билда). На обычной сборке 3.14 (GIL включён)
поведение итерации 2 не меняется — ускорения по-прежнему нет.